# <font color='teal'>Projet Data Collection</font>

| # | Source | Outil | Pages |
|---|---|---|---|
| 1 | [Books to Scrape](https://books.toscrape.com/catalogue/page-1.html) | requests + BeautifulSoup | 50 |
| 2 | [Gaaraas - Dakar auto](https://www.gaaraas.com/fr/users/dakar-auto?page=1) | requests + BeautifulSoup | 13 (les seules qui ont des annonces) |
| 3 | [Dakar-Auto - Motos & scooters](https://dakar-auto.com/senegal/motos-and-scooters-3) | requests + BeautifulSoup | 55 (erreur 404 ensuite) |
| 4 | Bulletins climatologiques 2020 et 2021 (PDF) | pdfplumber | 11 fichiers |

Pour chaque source : test sur un premier conteneur, boucle sur toutes les pages, nettoyage, puis stockage dans la base SQLite.

### <font color='bisque'>Importer les packages</font>

In [1]:
import time
import sqlite3
from pathlib import Path

import pandas as pd
from requests import get
from bs4 import BeautifulSoup as bs
import pdfplumber

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 40)

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

In [2]:
# base de données
conn = sqlite3.connect('data/projet_data_collection.db')

---
## <font color='teal'>1. Books to Scrape</font>
Variables : titre, prix, disponibilité, nombre de produits, note, nombre de reviews, description, catégorie, taxe.
Elles ne sont pas toutes sur la page liste : il faut ouvrir la page de chaque livre.

In [3]:
url = 'https://books.toscrape.com/catalogue/page-1.html'
page_source = get(url, headers=headers)
soup_page = bs(page_source.content, 'html.parser')
containers = soup_page.find_all('li', 'col-xs-6 col-sm-4 col-md-3 col-lg-3')
len(containers)

20

In [4]:
# premier livre
container = containers[0]
note = container.find('p', 'star-rating')['class'][1]
url_container = 'https://books.toscrape.com/catalogue/' + container.find('a')['href']
soup_container = bs(get(url_container, headers=headers).content, 'html.parser')
infos = {row.th.text: row.td.text for row in soup_container.find('table').find_all('tr')}
infos

{'UPC': 'a897fe39b1053632',
 'Product Type': 'Books',
 'Price (excl. tax)': '£51.77',
 'Price (incl. tax)': '£51.77',
 'Tax': '£0.00',
 'Availability': 'In stock (22 available)',
 'Number of reviews': '0'}

In [5]:
description = soup_container.find('div', id='product_description')
{
    'titre': soup_container.find('h1').text,
    'prix': soup_container.find('p', 'price_color').text,
    'disponibilite': infos['Availability'],
    'note': note,
    'nb_reviews': infos['Number of reviews'],
    'description': description.find_next_sibling('p').text[:80] + '...',
    'categorie': soup_container.find('ul', 'breadcrumb').find_all('li')[2].text.strip(),
    'taxe': infos['Tax'],
}

{'titre': 'A Light in the Attic',
 'prix': '£51.77',
 'disponibilite': 'In stock (22 available)',
 'note': 'Three',
 'nb_reviews': '0',
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic coll...",
 'categorie': 'Poetry',
 'taxe': '£0.00'}

In [6]:
# toutes les pages
df_books = pd.DataFrame()
for index_page in range(1, 51):
    url = f'https://books.toscrape.com/catalogue/page-{index_page}.html'
    soup_page = bs(get(url, headers=headers).content, 'html.parser')
    containers = soup_page.find_all('li', 'col-xs-6 col-sm-4 col-md-3 col-lg-3')
    data = []
    for container in containers:
        try:
            note = container.find('p', 'star-rating')['class'][1]
            url_container = 'https://books.toscrape.com/catalogue/' + container.find('a')['href']
            # 3 essais
            for essai in range(3):
                try:
                    soup_container = bs(get(url_container, headers=headers, timeout=30).content, 'html.parser')
                    break
                except:
                    time.sleep(2)
            infos = {row.th.text: row.td.text for row in soup_container.find('table').find_all('tr')}
            description = soup_container.find('div', id='product_description')
            dic = {
                'titre': soup_container.find('h1').text,
                'prix': soup_container.find('p', 'price_color').text,
                'disponibilite': infos.get('Availability'),
                'note': note,
                'nb_reviews': infos.get('Number of reviews'),
                'description': description.find_next_sibling('p').text if description else None,
                'categorie': soup_container.find('ul', 'breadcrumb').find_all('li')[2].text.strip(),
                'taxe': infos.get('Tax'),
                'url': url_container,
            }
            data.append(dic)
        except:
            print('livre ignoré :', url_container)
    DF = pd.DataFrame(data)
    df_books = pd.concat([df_books, DF], axis=0).reset_index(drop=True)
    if index_page % 10 == 0:
        print(f'page {index_page}/50 -> {df_books.shape[0]} livres')

page 10/50 -> 200 livres


page 20/50 -> 400 livres


page 30/50 -> 600 livres


page 40/50 -> 800 livres


page 50/50 -> 1000 livres


In [7]:
print(df_books.shape)
df_books.head()

(1000, 9)


,titre,prix,disponibilite,note,nb_reviews,description,categorie,taxe,url
0,A Light in the Attic,£51.77,In stock (22 available),Three,0,It's hard to imagine a world without...,Poetry,£0.00,https://books.toscrape.com/catalogue...
1,Tipping the Velvet,£53.74,In stock (20 available),One,0,"""Erotic and absorbing...Written with...",Historical Fiction,£0.00,https://books.toscrape.com/catalogue...
2,Soumission,£50.10,In stock (20 available),One,0,Dans une France assez proche de la n...,Fiction,£0.00,https://books.toscrape.com/catalogue...
3,Sharp Objects,£47.82,In stock (20 available),Four,0,"WICKED above her hipbone, GIRL acros...",Mystery,£0.00,https://books.toscrape.com/catalogue...
4,Sapiens: A Brief History of Humankind,£54.23,In stock (20 available),Five,0,From a renowned historian comes a gr...,History,£0.00,https://books.toscrape.com/catalogue...


### <font color='bisque'>Nettoyage des données Books</font>

In [8]:
books = df_books.copy()
# prix
books['prix'] = books['prix'].str.replace('£', '').str.replace('Â', '').astype(float)
books['taxe'] = books['taxe'].str.replace('£', '').str.replace('Â', '').astype(float)
# stock
books['nb_produits'] = books['disponibilite'].str.split('(').str[1].str.split(' ').str[0]
books['nb_produits'] = books['nb_produits'].fillna(0).astype(int)
books['disponibilite'] = books['disponibilite'].str.split('(').str[0].str.strip()
# note
books['note'] = books['note'].replace({'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}).astype(int)
books['nb_reviews'] = books['nb_reviews'].astype(int)
books['description'] = books['description'].fillna('').str.strip()
books = books.drop_duplicates(subset=['url']).reset_index(drop=True)
books = books[['titre', 'prix', 'disponibilite', 'nb_produits', 'note', 'nb_reviews',
               'description', 'categorie', 'taxe', 'url']]
books.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   titre          1000 non-null   str    
 1   prix           1000 non-null   float64
 2   disponibilite  1000 non-null   object 
 3   nb_produits    1000 non-null   int64  
 4   note           1000 non-null   int64  
 5   nb_reviews     1000 non-null   int64  
 6   description    1000 non-null   str    
 7   categorie      1000 non-null   str    
 8   taxe           1000 non-null   float64
 9   url            1000 non-null   str    
dtypes: float64(2), int64(3), object(1), str(4)
memory usage: 1.6+ MB


In [9]:
print(books.isna().sum().sum(), 'valeur(s) manquante(s)')
books.describe()

0 valeur(s) manquante(s)


,prix,nb_produits,note,nb_reviews,taxe
count,1000.00000,1000.000000,1000.000000,1000.0,1000.0
mean,35.07035,8.585000,2.923000,0.0,0.0
std,14.44669,5.654622,1.434967,0.0,0.0
min,10.00000,1.000000,1.000000,0.0,0.0
25%,22.10750,3.000000,2.000000,0.0,0.0
50%,35.98000,7.000000,3.000000,0.0,0.0
75%,47.45750,14.000000,4.000000,0.0,0.0
max,59.99000,22.000000,5.000000,0.0,0.0


In [10]:
books.to_csv('data/clean/books_clean.csv', index=False)
books.to_sql('books', conn, if_exists='replace', index=False)
pd.read_sql_query('SELECT categorie, COUNT(*) AS nb, ROUND(AVG(prix), 2) AS prix_moyen FROM books GROUP BY categorie ORDER BY nb DESC LIMIT 5', conn)

,categorie,nb,prix_moyen
0,Default,152,34.39
1,Nonfiction,110,34.26
2,Sequential Art,75,34.57
3,Add a comment,67,35.80
4,Fiction,65,36.07


---
## <font color='teal'>2. Gaaraas - Dakar auto (voitures)</font>
Variables : marque, modèle, année, prix, kilométrage, boîte de vitesses, région (+ carburant et statut).
Le sujet parle de 100 pages, mais l'annonceur n'a que 245 annonces, soit 13 pages : la boucle s'arrête à la première page vide.

In [11]:
url = 'https://www.gaaraas.com/fr/users/dakar-auto?page=1'
soup_page = bs(get(url, headers=headers).content, 'html.parser')
containers = soup_page.find_all('a', 'common-ad-card')
container = containers[0]
{
    'titre': container.find('h4').text,
    'prix': container.find('div', 'ad-vehicle-price').find('div', 'value').text,
    'kilometrage': container.find('div', 'ad-vehicle-mileage').find('div', 'value').text,
    'boite_vitesse': container.find('div', 'transmission').text,
    'region': container.find('div', 'location').text,
    'carburant': container.find('div', 'engine-capacity').text,
    'statut': container.find('div', 'ribbon').text if container.find('div', 'ribbon') else 'Disponible',
}

{'titre': '2010 Toyota RAV4',
 'prix': 'CFA 9 600 000',
 'kilometrage': '180 000 km',
 'boite_vitesse': 'Automatique',
 'region': 'Dakar',
 'carburant': 'N/A (Diesel)',
 'statut': 'Vendu'}

In [12]:
df_gaaraas = pd.DataFrame()
for index_page in range(1, 101):
    url = f'https://www.gaaraas.com/fr/users/dakar-auto?page={index_page}'
    page_source = get(url, headers=headers)
    if page_source.status_code != 200:
        print(f'page {index_page} : erreur {page_source.status_code}, à relancer')
        break
    soup_page = bs(page_source.content, 'html.parser')
    containers = soup_page.find_all('a', 'common-ad-card')
    if len(containers) == 0:
        print(f'page {index_page} vide -> arrêt')
        break
    data = []
    for container in containers:
        try:
            prix = container.find('div', 'ad-vehicle-price')
            kilometrage = container.find('div', 'ad-vehicle-mileage')
            dic = {
                'titre': container.find('h4').text,
                'prix': prix.find('div', 'value').text if prix else None,
                'kilometrage': kilometrage.find('div', 'value').text if kilometrage else None,
                'boite_vitesse': container.find('div', 'transmission').text,
                'region': container.find('div', 'location').text,
                'carburant': container.find('div', 'engine-capacity').text,
                'statut': container.find('div', 'ribbon').text if container.find('div', 'ribbon') else 'Disponible',
                'url': 'https://www.gaaraas.com' + container['href'],
            }
            data.append(dic)
        except:
            print('annonce ignorée, page', index_page)
    DF = pd.DataFrame(data)
    df_gaaraas = pd.concat([df_gaaraas, DF], axis=0).reset_index(drop=True)
    time.sleep(0.5)
print(df_gaaraas.shape)
df_gaaraas.head()

page 14 vide -> arrêt
(245, 8)


,titre,prix,kilometrage,boite_vitesse,region,carburant,statut,url
0,2010 Toyota RAV4,CFA 9 600 000,180 000 km,Automatique,Dakar,N/A (Diesel),Vendu,https://www.gaaraas.com/fr/vehicle_l...
1,2007 Renault Scenic,CFA 1 950 000,160 000 km,Manuelle,Dakar,N/A (Diesel),Vendu,https://www.gaaraas.com/fr/vehicle_l...
2,2004 Toyota Corolla,CFA 2 700 000,28 752 km,Manuelle,Dakar,N/A (Diesel),Vendu,https://www.gaaraas.com/fr/vehicle_l...
3,2007 Mercedes‒Benz 190,CFA 2 900 000,160 000 km,Automatique,Dakar,N/A (Essence),Vendu,https://www.gaaraas.com/fr/vehicle_l...
4,2007 Audi S3,CFA 2 000 000,120 000 km,Manuelle,Dakar,N/A (Diesel),Vendu,https://www.gaaraas.com/fr/vehicle_l...


### <font color='bisque'>Nettoyage des données Gaaraas</font>

In [13]:
gaaraas = df_gaaraas.copy()
gaaraas['titre'] = gaaraas['titre'].str.strip().str.replace('‒', '-').str.replace('–', '-')
# année, marque, modèle
avec_annee = gaaraas['titre'].str[:4].str.isnumeric()
gaaraas['annee'] = pd.to_numeric(gaaraas['titre'].str[:4], errors='coerce').astype('Int64')
nom = gaaraas['titre'].where(~avec_annee, gaaraas['titre'].str[5:])
gaaraas['marque'] = nom.str.split(' ').str[0]
gaaraas['modele'] = nom.str.split(' ', n=1).str[1].fillna('')
for marque in ['Land Rover', 'Alfa Romeo', 'Great Wall', 'Aston Martin', 'Rolls Royce']:
    ligne = nom.str.startswith(marque)
    gaaraas.loc[ligne, 'marque'] = marque
    gaaraas.loc[ligne, 'modele'] = nom[ligne].str[len(marque):].str.strip()
# prix et kilométrage
gaaraas['prix'] = gaaraas['prix'].str.replace('CFA', '').str.replace(' ', '').str.replace('\xa0', '')
gaaraas['prix'] = pd.to_numeric(gaaraas['prix'], errors='coerce')
gaaraas['kilometrage'] = gaaraas['kilometrage'].str.replace('km', '').str.replace(' ', '').str.replace('\xa0', '')
gaaraas['kilometrage'] = pd.to_numeric(gaaraas['kilometrage'], errors='coerce')
gaaraas.loc[gaaraas['kilometrage'] == 0, 'kilometrage'] = None
gaaraas['boite_vitesse'] = gaaraas['boite_vitesse'].str.strip()
gaaraas['region'] = gaaraas['region'].str.strip()
# carburant
gaaraas['carburant'] = gaaraas['carburant'].str.split('(').str[1].str.replace(')', '').str.strip()
gaaraas['statut'] = gaaraas['statut'].str.strip()
gaaraas = gaaraas.drop_duplicates(subset=['url']).reset_index(drop=True)
gaaraas = gaaraas[['marque', 'modele', 'annee', 'prix', 'kilometrage', 'boite_vitesse',
                   'region', 'carburant', 'statut', 'url']]
gaaraas.head()

,marque,modele,annee,prix,kilometrage,boite_vitesse,region,carburant,statut,url
0,Toyota,RAV4,2010,9600000.0,180000.0,Automatique,Dakar,Diesel,Vendu,https://www.gaaraas.com/fr/vehicle_l...
1,Renault,Scenic,2007,1950000.0,160000.0,Manuelle,Dakar,Diesel,Vendu,https://www.gaaraas.com/fr/vehicle_l...
2,Toyota,Corolla,2004,2700000.0,28752.0,Manuelle,Dakar,Diesel,Vendu,https://www.gaaraas.com/fr/vehicle_l...
3,Mercedes-Benz,190,2007,2900000.0,160000.0,Automatique,Dakar,Essence,Vendu,https://www.gaaraas.com/fr/vehicle_l...
4,Audi,S3,2007,2000000.0,120000.0,Manuelle,Dakar,Diesel,Vendu,https://www.gaaraas.com/fr/vehicle_l...


In [14]:
print(gaaraas.isna().sum())
gaaraas.describe()

marque           0
modele           0
annee            1
prix             1
kilometrage      4
boite_vitesse    0
region           0
carburant        0
statut           0
url              0
dtype: int64


,annee,prix,kilometrage
count,244.0,2.440000e+02,241.000000
mean,2007.479508,3.181148e+06,150368.651452
std,3.590624,2.056259e+06,52547.912479
min,1989.0,1.100000e+06,16000.000000
25%,2006.0,2.000000e+06,120000.000000
50%,2008.0,2.500000e+06,160000.000000
75%,2009.0,3.500000e+06,160000.000000
max,2015.0,2.000000e+07,500000.000000


In [15]:
gaaraas.to_csv('data/clean/gaaraas_clean.csv', index=False)
gaaraas.to_sql('gaaraas_voitures', conn, if_exists='replace', index=False)
pd.read_sql_query('SELECT marque, COUNT(*) AS nb FROM gaaraas_voitures GROUP BY marque ORDER BY nb DESC LIMIT 5', conn)

,marque,nb
0,Peugeot,65
1,Renault,34
2,Citroen,24
3,Toyota,21
4,Ford,17


---
## <font color='teal'>3. Dakar-Auto - Motos & scooters</font>
Consigne : *tout ce qui bouge*, sur 100 pages maximum. Le site n'a que 55 pages et renvoie une erreur 404 ensuite.
Le nom et le téléphone du vendeur ne sont pas collectés : ce sont des données personnelles.

In [16]:
url = 'https://dakar-auto.com/senegal/motos-and-scooters-3?page=1'
soup_page = bs(get(url, headers=headers).content, 'html.parser')
containers = soup_page.find_all('div', 'listing-card')
container = containers[0]
categorie = container.find('h4', 'listing-card__header__category')
attributs = [li.text.strip() for li in container.find_all('li', 'listing-card__attribute')]
print(len(containers), 'annonces sur la page')
print('titre      :', container.find('h2', 'listing-card__header__title').text.strip())
print('categorie  :', categorie.contents[0].strip(), '/', categorie.find('small').text)
print('prix       :', container.find('h3', 'listing-card__header__price').text.strip())
print('adresse    :', container.find('span', 'town-suburb').text, container.find('span', 'province').text)
print('attributs  :', attributs)

20 annonces sur la page
titre      : Honda X-ADV 2024
categorie  : Scooters / Occasion
prix       : 3 700 000 F CFA
adresse    : Virage, Dakar
attributs  : ['Ref. 140010', '1 km']


In [17]:
df_motos = pd.DataFrame()
for index_page in range(1, 101):
    url = f'https://dakar-auto.com/senegal/motos-and-scooters-3?page={index_page}'
    page_source = get(url, headers=headers)
    if page_source.status_code != 200:
        print(f'page {index_page} : erreur {page_source.status_code} -> arrêt')
        break
    soup_page = bs(page_source.content, 'html.parser')
    containers = soup_page.find_all('div', 'listing-card')
    data = []
    for container in containers:
        try:
            categorie = container.find('h4', 'listing-card__header__category')
            attributs = [li.text.strip() for li in container.find_all('li', 'listing-card__attribute')]
            kilometrage = [a for a in attributs if a.endswith('km')]
            dic = {
                'titre': container.find('h2', 'listing-card__header__title').text,
                'categorie': categorie.contents[0],
                'etat': categorie.find('small').text if categorie.find('small') else None,
                'prix': container.find('h3', 'listing-card__header__price').text,
                'quartier': container.find('span', 'town-suburb').text if container.find('span', 'town-suburb') else None,
                'ville': container.find('span', 'province').text,
                'kilometrage': kilometrage[0] if kilometrage else None,
                'reference': attributs[0] if attributs else None,
                'publication': container.find('p', 'time').text,
                'url': 'https://dakar-auto.com' + container.find('h2').find('a')['href'],
            }
            data.append(dic)
        except:
            print('annonce ignorée, page', index_page)
    DF = pd.DataFrame(data)
    df_motos = pd.concat([df_motos, DF], axis=0).reset_index(drop=True)
    time.sleep(0.5)
print(df_motos.shape)
df_motos.head(3)

page 56 : erreur 404 -> arrêt
(1089, 10)


,titre,categorie,etat,prix,quartier,ville,kilometrage,reference,publication,url
0,\n\n\t\t\tHonda X-ADV 2024\t\t\n,\n\t\tScooters\t\t\t\t\t,Occasion,\n\t\t\t\t\t3 700 000 F CFA,"Virage,",Dakar,1 km,Ref. 140010,"\n\t\tPassée, il y a 9 mois\t",https://dakar-auto.com/senegal/motos...
1,\n\n\t\t\tHonda SH 2010\t\t\n,\n\t\tMotos\t\t\t\t\t,Occasion,\n\t\t\t\t\t450 F CFA,"Guédiawaye,",Dakar,160 km,Ref. 139931,"\n\t\tPassée, il y a 12 mois\t",https://dakar-auto.com/senegal/motos...
2,\n\n\t\t\tHonda CBF 2007\t\t\n,\n\t\tMotos\t\t\t\t\t,Occasion,\n\t\t\t\t\t1 500 000 F CFA,"Parcelles Assainies,",Dakar,11000 km,Ref. 139922,"\n\t\tPassée, il y a 12 mois\t",https://dakar-auto.com/senegal/motos...


### <font color='bisque'>Nettoyage des données Motos</font>

In [18]:
motos = df_motos.copy()
# espaces en trop
for col in ['titre', 'categorie', 'etat', 'prix', 'quartier', 'ville', 'kilometrage', 'reference', 'publication']:
    motos[col] = motos[col].str.split().str.join(' ')
# année, marque, modèle
mots = motos['titre'].str.split(' ')
dernier = mots.str[-1]
motos['annee'] = pd.to_numeric(dernier.where(dernier.str.len() == 4), errors='coerce').astype('Int64')
a_enlever = (dernier == '0') | motos['annee'].notna()
nom = mots.where(~a_enlever, mots.str[:-1]).str.join(' ')
motos['marque'] = nom.str.split(' ').str[0].replace({'Bmw': 'BMW'})
motos['modele'] = nom.str.split(' ', n=1).str[1]
# prix
motos['prix'] = pd.to_numeric(motos['prix'].str.replace('F CFA', '').str.replace(' ', ''), errors='coerce')
motos.loc[motos['prix'] < 10000, 'prix'] = None
# kilométrage
motos['kilometrage'] = pd.to_numeric(motos['kilometrage'].str.replace('km', '').str.replace(' ', ''), errors='coerce')
motos.loc[(motos['etat'] == 'Occasion') & (motos['kilometrage'] <= 10), 'kilometrage'] = None
motos['kilometrage'] = motos['kilometrage'].astype('Int64')
motos['reference'] = motos['reference'].str.replace('Ref.', '').str.strip()
motos['quartier'] = motos['quartier'].str.rstrip(',')
motos['publication'] = motos['publication'].str.replace('Passée, ', '')
motos = motos.drop_duplicates(subset=['reference']).reset_index(drop=True)
motos = motos[['marque', 'modele', 'annee', 'categorie', 'etat', 'prix', 'kilometrage',
               'quartier', 'ville', 'reference', 'publication', 'url']]
motos.head()

,marque,modele,annee,categorie,etat,prix,kilometrage,quartier,ville,reference,publication,url
0,Honda,X-ADV,2024,Scooters,Occasion,3700000.0,<NA>,Virage,Dakar,140010,il y a 9 mois,https://dakar-auto.com/senegal/motos...
1,Honda,SH,2010,Motos,Occasion,NaN,160,Guédiawaye,Dakar,139931,il y a 12 mois,https://dakar-auto.com/senegal/motos...
2,Honda,CBF,2007,Motos,Occasion,1500000.0,11000,Parcelles Assainies,Dakar,139922,il y a 12 mois,https://dakar-auto.com/senegal/motos...
3,SYM,125S,2023,Motos,Occasion,620000.0,1200,Rufisque,Dakar,139902,il y a 1 année,https://dakar-auto.com/senegal/motos...
4,Yamaha,TMax,2023,Scooters,Occasion,4300000.0,<NA>,VDN,Dakar,139869,il y a 1 année,https://dakar-auto.com/senegal/motos...


In [19]:
print(motos.isna().sum())
motos.describe()

marque           0
modele           0
annee          360
categorie        0
etat             0
prix           108
kilometrage    632
quartier         0
ville            0
reference        0
publication      0
url              0
dtype: int64


,annee,prix,kilometrage
count,718.0,9.700000e+02,446.0
mean,2011.713092,1.313352e+06,25322.560538
std,6.764096,1.336501e+06,38316.684367
min,1989.0,7.000000e+04,1.0
25%,2008.0,4.300000e+05,3598.0
50%,2012.0,8.000000e+05,15000.0
75%,2017.0,1.700000e+06,32000.0
max,2025.0,1.400000e+07,452125.0


In [20]:
motos.to_csv('data/clean/motos_clean.csv', index=False)
motos.to_sql('dakar_auto_motos', conn, if_exists='replace', index=False)
pd.read_sql_query('SELECT categorie, etat, COUNT(*) AS nb, ROUND(AVG(prix)) AS prix_moyen FROM dakar_auto_motos GROUP BY categorie, etat', conn)

,categorie,etat,nb,prix_moyen
0,Motos,Neuf,32,2553824.0
1,Motos,Occasion,949,1308310.0
2,Scooters,Neuf,7,664286.0
3,Scooters,Occasion,90,1171395.0


---
## <font color='teal'>4. Bulletins climatologiques (PDF) - pdfplumber</font>
Le tableau des températures par station est sur la dernière page (page 8) de chaque bulletin.
On ajoute l'année (nom du dossier) et le mois (nom du fichier).

In [21]:
# liste des bulletins
fichiers = sorted(Path('data/bulletins').glob('*/*.pdf'))
fichiers

[WindowsPath('data/bulletins/2020/climato_avril.pdf'),
 WindowsPath('data/bulletins/2020/climato_decembre.pdf'),
 WindowsPath('data/bulletins/2020/climato_fevrier.pdf'),
 WindowsPath('data/bulletins/2020/climato_janvier.pdf'),
 WindowsPath('data/bulletins/2020/climato_mai.pdf'),
 WindowsPath('data/bulletins/2020/climato_mars.pdf'),
 WindowsPath('data/bulletins/2020/climato_novembre.pdf'),
 WindowsPath('data/bulletins/2021/climato_avril.pdf'),
 WindowsPath('data/bulletins/2021/climato_fevrier.pdf'),
 WindowsPath('data/bulletins/2021/climato_janvier.pdf'),
 WindowsPath('data/bulletins/2021/climato_mai.pdf')]

In [22]:
# fichier test
with pdfplumber.open(fichiers[0]) as pdf:
    page = pdf.pages[7]
    tables = page.extract_tables()
print(len(tables), 'tableau(x) sur la page')
tables[0][:5]

1 tableau(x) sur la page


[['Stations', 'Tmax (°C)', None, 'Tmin (°C)', None],
 [None, 'Maxmax', 'Minmax', 'Maxmin', 'Minmin'],
 ['Bakel', '46,2', '39', '32', '25,2'],
 ['Bambey', '44,5', '29,5', '25', '19,6'],
 ['Cap', '39,8', '26,9', '23', '18,6']]

In [23]:
# nouvelles colonnes
colonnes = ['station', 'tmax_maxmax', 'tmax_minmax', 'tmin_maxmin', 'tmin_minmin']

# tous les fichiers
df_bulletins = pd.DataFrame()
for fichier in fichiers:
    with pdfplumber.open(fichier) as pdf:
        table = pdf.pages[7].extract_tables()[0]
    DF = pd.DataFrame(table[2:], columns=colonnes)
    DF['annee'] = int(fichier.parent.name)
    DF['mois'] = fichier.stem.replace('climato_', '')
    df_bulletins = pd.concat([df_bulletins, DF], axis=0).reset_index(drop=True)
print(df_bulletins.shape)
df_bulletins.head()

(258, 7)


,station,tmax_maxmax,tmax_minmax,tmin_maxmin,tmin_minmin,annee,mois
0,Bakel,"46,2",39,32,"25,2",2020,avril
1,Bambey,"44,5","29,5",25,"19,6",2020,avril
2,Cap,"39,8","26,9",23,"18,6",2020,avril
3,Dakar,"39,5",24,21,18,2020,avril
4,Diourbel,"45,5","33,2","26,7",19,2020,avril


In [24]:
df_bulletins.info()

<class 'pandas.DataFrame'>
RangeIndex: 258 entries, 0 to 257
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   station      258 non-null    str  
 1   tmax_maxmax  258 non-null    str  
 2   tmax_minmax  258 non-null    str  
 3   tmin_maxmin  258 non-null    str  
 4   tmin_minmin  258 non-null    str  
 5   annee        258 non-null    int64
 6   mois         258 non-null    str  
dtypes: int64(1), str(6)
memory usage: 20.8 KB


### <font color='bisque'>Nettoyage des données Bulletins</font>

In [25]:
bulletins = df_bulletins.copy()
# noms des stations
print(sorted(bulletins['station'].unique()))

['Bakel', 'Bambey', 'Cap', 'Dakar', 'Diass', 'Diourbel', 'Fatick', 'Goudiry', 'Kaolack', 'Kedougou', 'Kolda', 'Koungheul', 'Kédougou', 'Linguere', 'Linguère', 'Louga', 'Matam', 'Mbour', 'Nioro', 'Podor', 'Ranerou', 'Ranérou', 'Saint Louis', 'Simenti', 'Stlouis', 'Tamba', 'Tambacounda', 'Thies', 'Thiès', 'Vélingara', 'Ziguinchor']


In [26]:
noms_stations = {'Cap': 'Cap Skirring', 'Tamba': 'Tambacounda', 'Kedougou': 'Kédougou',
                 'Linguere': 'Linguère', 'Ranerou': 'Ranérou', 'Stlouis': 'Saint-Louis',
                 'Saint Louis': 'Saint-Louis', 'Thies': 'Thiès', 'Nioro': 'Nioro du Rip'}
bulletins['station'] = bulletins['station'].str.strip().replace(noms_stations)

# conversion
temperatures = ['tmax_maxmax', 'tmax_minmax', 'tmin_maxmin', 'tmin_minmin']
for col in temperatures:
    bulletins[col] = pd.to_numeric(bulletins[col].str.replace(',', '.'), errors='coerce')
bulletins.dtypes

station            str
tmax_maxmax    float64
tmax_minmax    float64
tmin_maxmin    float64
tmin_minmin    float64
annee            int64
mois               str
dtype: object

Certaines cases contiennent **« NP »** : *Non Parvenu* (légende du bulletin de février 2020).
`errors='coerce'` les transforme en valeurs manquantes.

In [27]:
# vérifier les valeurs manquantes
print(bulletins[temperatures].isna().sum())
# valeurs aberrantes
aberrant = (bulletins['tmax_minmax'] < bulletins['tmin_minmin']) | (bulletins['tmax_minmax'] > bulletins['tmax_maxmax'])
bulletins[aberrant | (bulletins['tmin_maxmin'] > 35)]

tmax_maxmax    10
tmax_minmax    10
tmin_maxmin    14
tmin_minmin    14
dtype: int64


,station,tmax_maxmax,tmax_minmax,tmin_maxmin,tmin_minmin,annee,mois
57,Koungheul,43.6,10.0,27.2,15.5,2020,fevrier
72,Bambey,39.5,2.9,24.0,14.8,2020,janvier
216,Fatick,38.8,29.0,38.5,14.0,2021,janvier


In [28]:
bulletins.loc[aberrant, 'tmax_minmax'] = None
bulletins.loc[bulletins['tmin_maxmin'] > 35, 'tmin_maxmin'] = None
manquait = bulletins[temperatures].isna()

# médiane de la station (au moins 6 mesures)
for col in temperatures:
    for station in bulletins['station'].unique():
        ligne = bulletins['station'] == station
        if bulletins.loc[ligne, col].count() >= 6:
            bulletins.loc[ligne, col] = bulletins.loc[ligne, col].fillna(bulletins.loc[ligne, col].median())
bulletins['valeur_imputee'] = (manquait & bulletins[temperatures].notna()).any(axis=1).astype(int)

mois = ['janvier', 'fevrier', 'mars', 'avril', 'mai', 'juin', 'juillet',
        'aout', 'septembre', 'octobre', 'novembre', 'decembre']
bulletins['mois_num'] = bulletins['mois'].apply(lambda m: mois.index(m) + 1)
bulletins = bulletins.sort_values(['annee', 'mois_num', 'station']).reset_index(drop=True)
bulletins = bulletins[['annee', 'mois', 'mois_num', 'station'] + temperatures + ['valeur_imputee']]
print(bulletins[temperatures].isna().sum())
bulletins.describe()

tmax_maxmax     6
tmax_minmax     6
tmin_maxmin    10
tmin_minmin    10
dtype: int64


,annee,mois_num,tmax_maxmax,tmax_minmax,tmin_maxmin,tmin_minmin,valeur_imputee
count,258.000000,258.000000,252.000000,252.000000,248.000000,248.000000,258.000000
mean,2020.368217,4.581395,41.789286,32.028373,25.011492,16.745565,0.027132
std,0.483258,3.583789,3.515246,4.663824,3.216518,3.315154,0.162783
min,2020.000000,1.000000,27.500000,18.500000,17.000000,7.800000,0.000000
25%,2020.000000,2.000000,39.975000,28.975000,22.575000,14.100000,0.000000
50%,2020.000000,4.000000,41.550000,32.250000,24.950000,16.600000,0.000000
75%,2021.000000,5.000000,44.500000,35.225000,27.000000,19.000000,0.000000
max,2021.000000,12.000000,48.400000,43.100000,33.500000,26.600000,1.000000


In [29]:
bulletins.to_csv('data/clean/bulletins_clean.csv', index=False)
bulletins.to_sql('bulletins_climato', conn, if_exists='replace', index=False)
pd.read_sql_query('SELECT annee, mois, ROUND(AVG(tmax_maxmax), 1) AS tmax_moyenne FROM bulletins_climato GROUP BY annee, mois_num ORDER BY annee, mois_num', conn)

,annee,mois,tmax_moyenne
0,2020,janvier,40.2
1,2020,fevrier,41.5
2,2020,mars,43.5
3,2020,avril,44.8
4,2020,mai,44.1
5,2020,novembre,40.5
6,2020,decembre,38.2
7,2021,janvier,39.4
8,2021,fevrier,40.2
9,2021,avril,43.7


---
## <font color='teal'>5. Vérification de la base de données</font>

In [30]:
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)['name']
for t in tables:
    print(t, ':', pd.read_sql_query(f'SELECT COUNT(*) AS nb FROM {t}', conn)['nb'][0], 'lignes')

books : 1000 lignes
gaaraas_voitures : 245 lignes
dakar_auto_motos : 1078 lignes
bulletins_climato : 258 lignes


In [31]:
conn.close()